# Hybrid Mamba-CNN: Spatial Convolution and State-Space Model Fusion for Medical Image Classification

## Abstract & Architecture Overview
This notebook implements the training and validation pipeline for the Hybrid Mamba-CNN model on the **ISIC 2019** skin lesion benchmark dataset. The architecture integrates continuous local feature representations extracted via Convolutional Neural Network (CNN) backbones with long-range sequence modeling capabilities provided by Selective State-Space Models (Vision Mamba). Cross-modal feature representations are dynamically merged using an attention-guided fusion mechanism prior to final multi-class diagnostic prediction.

### Key Methodological Components
1. **Convolutional Feature Extraction**: ResNet-50 backbone extracting localized spatial texture maps.
2. **State-Space Feature Modeling**: Vision Mamba (Vim) blocks modeling global contextual dependency.
3. **Attention Fusion Module**: Adaptive cross-attention weighting of localized and sequence features.
4. **Class Imbalance Mitigation**: Class-weighted Focal Loss and stratified train/validation/test partitioning.
5. **Optimization Strategy**: AdamW optimizer with Cosine Annealing learning rate scheduling and early stopping based on macro F1 score.

## 1. Environment & Library Initialization
Import required Python scientific computing packages, PyTorch deep learning modules, and local utility libraries.

In [ ]:
import os
import sys
import time
import random
from pathlib import Path
from typing import Dict, Any, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Ensure project root directory is present in system path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.config import ConfigManager
from utils.dataset import create_dataloaders
from models.hybrid_model import create_hybrid_model
from models.cnn_baseline import create_cnn_model
from models.vim_baseline import create_vim_model
from training.engine import create_trainer

## 2. Reproducibility & Hardware Configuration
Configure fixed random seeds for deterministic execution across PyTorch, NumPy, and CUDA subroutines, and inspect available compute hardware.

In [ ]:
def initialize_reproducibility(seed: int = 42) -> None:
    """Configure deterministic seeds across all random number generators."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
initialize_reproducibility(SEED)

if torch.cuda.is_available():
    device = torch.device('cuda')
    device_name = torch.cuda.get_device_name(0)
    device_count = torch.cuda.device_count()
    print(f"Compute Target : GPU ({device_name})")
    print(f"Available GPUs : {device_count}")
else:
    device = torch.device('cpu')
    print("Compute Target : CPU (CUDA acceleration unavailable)")

## 3. Experiment Configuration Management
Load hyperparameter definitions from the experimental specification YAML file and set relative workspace paths.

In [ ]:
config_rel_path = project_root / 'configs' / 'hybrid_config.yaml'
if not config_rel_path.exists():
    config_rel_path = project_root / 'configs' / 'base_config.yaml'

config_manager = ConfigManager(str(config_rel_path))
config = config_manager.to_dict()

# Override data directory path to match environment root
config['dataset']['data_dir'] = str(project_root / 'data')

print("==================================================")
print("EXPERIMENT HYPERPARAMETER SPECIFICATION")
print("==================================================")
print(f"Experiment Name    : {config['experiment']['name']}")
print(f"Model Architecture : {config['model']['type']}")
print(f"CNN Backbone       : {config['model']['cnn']['backbone']}")
print(f"Fusion Strategy    : {config['model']['fusion']['method']}")
print(f"Input Resolution   : {config['dataset']['img_size']}x{config['dataset']['img_size']}")
print(f"Number of Classes  : {config['dataset']['num_classes']}")
print(f"Batch Size         : {config['training']['batch_size']}")
print(f"Target Epochs      : {config['training']['num_epochs']}")
print(f"Initial Learning Rate: {config['training']['learning_rate']}")
print(f"Loss Function      : {config['training']['loss']['type']}")
print("==================================================")

## 4. Dataset Partitioning & DataLoader Construction
Instantiate PyTorch Dataset and DataLoader objects for training, validation, and testing partitions with class-stratified splits.

In [ ]:
train_loader, val_loader, test_loader, class_names = create_dataloaders(
    data_dir=config['dataset']['data_dir'],
    batch_size=config['training']['batch_size'],
    num_workers=config['hardware']['num_workers'],
    img_size=config['dataset']['img_size']
)

print("Dataset Partition Statistics:")
print(f"  Training Partition   : {len(train_loader.dataset):>6} samples ({len(train_loader):>4} batches)")
print(f"  Validation Partition : {len(val_loader.dataset):>6} samples ({len(val_loader):>4} batches)")
print(f"  Testing Partition    : {len(test_loader.dataset):>6} samples ({len(test_loader):>4} batches)")
print(f"  Target Diagnostic Classes ({len(class_names)}): {class_names}")

## 5. Input Batch Statistics & Tensor Validation
Inspect sample tensor dimensions, data types, and value intervals from a representative training minibatch.

In [ ]:
sample_batch = next(iter(train_loader))
images_tensor = sample_batch['image']
labels_tensor = sample_batch['label']

print("Minibatch Tensor Specification:")
print(f"  Image Tensor Shape   : {list(images_tensor.shape)}")
print(f"  Image Tensor Dtype   : {images_tensor.dtype}")
print(f"  Image Value Interval : [{images_tensor.min():.3f}, {images_tensor.max():.3f}]")
print(f"  Label Tensor Shape   : {list(labels_tensor.shape)}")
print(f"  Label Tensor Dtype   : {labels_tensor.dtype}")
print(f"  Batch Label Samples  : {labels_tensor[:8].tolist()}")

## 6. Visual Inspection of Preprocessed Lesion Samples
Render representative augmented dermatoscopic images from the training dataset alongside ground-truth diagnostic category labels.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7), dpi=100)
for idx, ax in enumerate(axes.flat):
    if idx < len(images_tensor):
        img_np = images_tensor[idx].permute(1, 2, 0).numpy()
        # Rescale for visual plotting
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
        cls_idx = labels_tensor[idx].item()
        cls_label = class_names[cls_idx] if cls_idx < len(class_names) else f"Class {cls_idx}"
        
        ax.imshow(img_np)
        ax.set_title(f"{cls_label} (Index: {cls_idx})", fontsize=10, fontweight='bold')
        ax.axis('off')

plt.suptitle('ISIC 2019 Preprocessed & Augmented Sample Minibatch', fontsize=13, y=0.98)
plt.tight_layout()
plt.show()

## 7. Model Construction & Parameter Analysis
Instantiate the Hybrid Mamba-CNN neural network and compute parameter allocation across sub-modules.

In [ ]:
model = create_hybrid_model(config).to(device)

total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_trainable_parameters = total_parameters - trainable_parameters

print("==================================================")
print("MODEL PARAMETER ALLOCATION")
print("==================================================")
print(f"Total Parameters         : {total_parameters:>12,}")
print(f"Trainable Parameters     : {trainable_parameters:>12,}")
print(f"Non-Trainable Parameters : {non_trainable_parameters:>12,}")
print("==================================================")

## 8. Forward Pass Dimensionality Verification
Execute a mock forward pass using synthetic tensor inputs to confirm feature extraction, attention fusion, and logit output shape alignment.

In [ ]:
dummy_input = torch.randn(
    2, 3, config['dataset']['img_size'], config['dataset']['img_size'],
    device=device
)

model.eval()
with torch.no_grad():
    dummy_output = model(dummy_input)
    if isinstance(dummy_output, dict):
        logits = dummy_output['logits']
    else:
        logits = dummy_output

print("Forward Pass Verification:")
print(f"  Input Dimension  : {list(dummy_input.shape)}")
print(f"  Output Logits    : {list(logits.shape)}")
print(f"  Expected Classes : {config['dataset']['num_classes']}")
assert logits.shape == (2, config['dataset']['num_classes']), "Output logit shape mismatch!"

## 9. Training Engine & Optimization Setup
Initialize the `TrainingEngine` module managing loss calculation, backpropagation, optimizer states, learning rate scheduling, and model checkpointing.

In [ ]:
trainer = create_trainer(
    config=config,
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device
)

print("Training Engine Initialized:")
print(f"  Optimizer        : {type(trainer.optimizer).__name__}")
print(f"  Loss Function    : {type(trainer.criterion).__name__}")
print(f"  Early Stopping   : Patience = {trainer.patience} epochs")
print(f"  Checkpoint Dir   : {trainer.checkpoint_dir}")

## 10. Epoch Training Loop Execution
Execute the complete multi-epoch training pipeline, tracking loss minimization, accuracy convergence, and validation macro F1 performance.

In [ ]:
print("Starting model training process...")
training_results = trainer.train()
print("Model training process completed successfully.")

## 11. Quantitative Convergence & Learning Dynamics
Plot cross-epoch progression of training and validation loss, accuracy, and macro F1 metrics.

In [ ]:
history = training_results.get('training_history', [])

if history:
    epochs = [entry['epoch'] for entry in history]
    train_loss = [entry['train']['loss'] for entry in history]
    val_loss = [entry['val']['loss'] for entry in history]
    train_acc = [entry['train']['accuracy'] for entry in history]
    val_acc = [entry['val']['accuracy'] for entry in history]
    train_f1 = [entry['train']['f1_macro'] for entry in history]
    val_f1 = [entry['val']['f1_macro'] for entry in history]

    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=100)

    # Subplot 1: Loss Progression
    axes[0].plot(epochs, train_loss, label='Training Loss', color='#1f77b4', linewidth=2, marker='o', markersize=4)
    axes[0].plot(epochs, val_loss, label='Validation Loss', color='#ff7f0e', linewidth=2, marker='s', markersize=4)
    axes[0].set_title('Cross-Entropy Loss Progression', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch', fontsize=10)
    axes[0].set_ylabel('Loss', fontsize=10)
    axes[0].legend(frameon=True, facecolor='white', edgecolor='none')
    axes[0].grid(True, linestyle='--', alpha=0.5)

    # Subplot 2: Accuracy Progression
    axes[1].plot(epochs, train_acc, label='Training Accuracy', color='#2ca02c', linewidth=2, marker='o', markersize=4)
    axes[1].plot(epochs, val_acc, label='Validation Accuracy', color='#d62728', linewidth=2, marker='s', markersize=4)
    axes[1].set_title('Top-1 Accuracy Progression', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch', fontsize=10)
    axes[1].set_ylabel('Accuracy', fontsize=10)
    axes[1].legend(frameon=True, facecolor='white', edgecolor='none')
    axes[1].grid(True, linestyle='--', alpha=0.5)

    # Subplot 3: Macro F1 Progression
    axes[2].plot(epochs, train_f1, label='Training F1 Macro', color='#9467bd', linewidth=2, marker='o', markersize=4)
    axes[2].plot(epochs, val_f1, label='Validation F1 Macro', color='#8c564b', linewidth=2, marker='s', markersize=4)
    axes[2].set_title('Macro F1 Score Progression', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Epoch', fontsize=10)
    axes[2].set_ylabel('F1 Score', fontsize=10)
    axes[2].legend(frameon=True, facecolor='white', edgecolor='none')
    axes[2].grid(True, linestyle='--', alpha=0.5)

    plt.suptitle('Hybrid Mamba-CNN Model Learning Curves', fontsize=14, y=1.02, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 12. Final Test Set Evaluation & Benchmark Reporting
Evaluate the best model checkpoint on the independent test dataset partition and summarize quantitative classification metrics.

In [ ]:
test_metrics = training_results.get('test_metrics', {})

if test_metrics:
    print("==================================================")
    print("FINAL HELD-OUT TEST EVALUATION METRICS")
    print("==================================================")
    print(f"Test Accuracy        : {test_metrics.get('accuracy', 0.0):.4f}")
    print(f"Test F1 (Macro)      : {test_metrics.get('f1_macro', 0.0):.4f}")
    print(f"Test F1 (Weighted)   : {test_metrics.get('f1_weighted', 0.0):.4f}")
    print(f"Test ROC-AUC         : {test_metrics.get('auc', 0.0):.4f}")
    print(f"Test Precision (W)   : {test_metrics.get('precision_weighted', 0.0):.4f}")
    print(f"Test Recall (W)      : {test_metrics.get('recall_weighted', 0.0):.4f}")
    print("==================================================")